# Auto fixing with model feedback

## Fundamental Concept

The self correction represents a mechanism that allow LLMs to enhance their answers throughout iterative cycles of feedback, building a dynamic learning environment during their execution

# Implementing

In [ ]:
@agent.tool(retries=3) # Defining a maximum number of tries
async def validate_answer(ctx, data: dict) -> bool:
    if not is_valid(data):
        # Provides a specific feedback to the model
        raise ModelRetry("The data is not valid, please check the required arguments!")
    return True

## Practical benefits

Granular Control: You define which criteria must be followed
Educational Feedback: The model "learns" with the specific orientation

## Feedback Strategies and Best Practices

1. **Be specific**: Say exactly what is missing or incorrect.
2. **Provide context**: Explain why the correction is needed
3. **Suggest approaches**: Provide a clear direction to the solution
4. **Limit the retries**: Avoid infinite loops with retries=N

In [5]:
from pydantic import BaseModel
from pydantic_ai import Agent, RunContext, ModelRetry
from dotenv import load_dotenv

load_dotenv()

class ImcResult(BaseModel):
    imc: float
    status: str
    
agent = Agent(
    model='openrouter:google/gemini-2.5-flash',
    output_type=ImcResult,
    system_prompt="""
    You are a health assistant. Your task is to calculate the IMC of a person based on their height and weight.
    Call the 'calculate_imc' tool to get the IMC.
    The status is determined by the following ranges:
    - Underweight: less than 18.5
    - Normal weight: between 18.5 and 24.9
    - Overweight: between 25 and 29.9
    - Obesity: 30 or higher
    """
)

agent.tool(retries=3)
async def calculate_imc(ctx: RunContext, height: int, weight: float) -> float:
    """
    Calculate the IMC of a person based on their height and weight.

    Args:
        ctx (RunContext): The context of the run
        height (int): The height of the person in centimeters
        weight (float): The weight of the person in kilograms

    Returns:
        float: The IMC of the person
    """
    if (height <= 0 or weight <= 0):
        raise ModelRetry("The height and weight must be greater than 0")
    return weight / (height ** 2)

async def calculate_imc_with_feedback(height: int, weight: float) -> ImcResult:
    result = await agent.run(
        f"Calculate the IMC of a person with height {height}cm and weight {weight}kg"
    )
    return result.output

result = await calculate_imc_with_feedback(183, 86)
print('First result: ', result)

result = await calculate_imc_with_feedback(-1, 86)
print('Second result: ', result)

result = await calculate_imc_with_feedback(183, -2)
print('Third result: ', result)

First result:  imc=25.74 status='Overweight'
Second result:  imc=0.0 status='Invalid input: Height and weight must be positive numbers.'
Third result:  imc=-0.597282717909062 status='Underweight'
